In [1]:
import ollama 
import os
from tqdm import tqdm
import json
import signal
import argparse
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
import re

import sys
from collections import defaultdict

In [5]:
sys.argv = [
    'notebook',  
    '--modelname', 'llava-llama3:8b',
    '--data', 'gully',
    '--data_path','/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline/reason_first_results_llava-llama3:8b.json',
    '--subset', 'train',
    '--results_dir', '/mnt/jacket/WACV-2025-Workshop-ViGIR/results/baseline',
    '--timeout', '20',
    '--model_unloading'
]

In [6]:
parser = argparse.ArgumentParser(description="A script to evaluate V-LLMs on different image classification datasets")

parser.add_argument("--modelname", type=str, required=True, help="The name of the V-LLM model")
parser.add_argument("--data", type=str, required=True, help="Dataset name")
parser.add_argument("--data_path", type=str, required=True, help="Path to the image data dir")
parser.add_argument("--subset", type=str, required=True, help="train, test or validation set")
parser.add_argument("--results_dir", type=str, required=True, help="Folder name to save results")
parser.add_argument("--timeout", type=int, default=40, help="time out duration to skip one sample")
parser.add_argument("--model_unloading", action="store_true", help="Enables unloading mode. Every 100 samples it unloades the model from the GPU to avoid carshing.")

args = parser.parse_args()

In [7]:
# Load test set:
file_path = os.path.join(args.data_path)
with open(file_path, 'r') as file:
    data = json.load(file)

print('Number of Annotated GT Images: ', len(data.keys()))
data_keys_test = list(data.keys())
print('Number of Annotated GT Images (List): ', len(data_keys_test))

Number of Annotated GT Images:  311
Number of Annotated GT Images (List):  311


In [8]:
data

{'100': ['/root/home/data/neg_collage_100.jpg',
  'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. For example, some images may show more trees or less trees due to seasonal changes, weather patterns, or human activity. By comparing the images side by side, one can observe these differences and identify any ephemeral gully appearances that were present at different times but not consistently across all images.'],
 '1004': ['/root/home/data/neg_collage_1004.jpg',
  'Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. These changes could be due to factors such as weather patterns, soil erosion, or human activity. By comparing the images side by

In [9]:
data_prompt = {}
for key, item in data.items():
    prompt = "\nA:" + item[-1]
    data_prompt[key] = [prompt]

In [10]:
data_prompt

{'100': ['\nA:Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. For example, some images may show more trees or less trees due to seasonal changes, weather patterns, or human activity. By comparing the images side by side, one can observe these differences and identify any ephemeral gully appearances that were present at different times but not consistently across all images.'],
 '1004': ['\nA:Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. These changes could be due to factors such as weather patterns, soil erosion, or human activity. By comparing the images side by side, one can observe how the area has evolved over time, revealing any te

In [11]:
model_name = args.modelname
ollama.pull(model_name)

timeout_duration = args.timeout

options= {  # new
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048, # must be set, otherwise slightly random output
        }

model_labels = {}
count = 0

In [12]:
count = 0
saving_response=copy.deepcopy(data_prompt)

for key, info in tqdm(data_prompt.items()):

    #print(key)
    #print(info[0])
    #sys.exit()
    question = "Answer only with yes or no, does the following response suggest that there is an ephemeral gully:\n\n"
    question += info[0]
    #question += "\n\nAfter considering these responses, provide a clear conclusion: Is there evidence of an ephemeral gully? Answer with yes or no only."
    
    #print(question)
    #sys.exit()
        
    count+=1
    #for question in questions:
    response = ollama.generate(model=model_name, 
                               prompt=question, 
                               #images=info, 
                               options=options)
    saving_response[key].append(response['response'])
    print(response['response'])

    #sys.exit()
        

  1%|▉                                                                                                     | 3/311 [00:03<05:20,  1.04s/it]

Yes
Yes
Yes


  2%|█▋                                                                                                    | 5/311 [00:04<02:46,  1.83it/s]

Yes
Yes
Yes


  3%|██▉                                                                                                   | 9/311 [00:04<01:14,  4.06it/s]

...

Yes
Yes
Yes


 99%|██████████████████████████████████████████████████████████████████████████████████████████████████▋ | 307/311 [00:29<00:00, 12.27it/s]

Yes
Yes
Yes


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 311/311 [00:29<00:00, 10.57it/s]

Yes
Yes
Yes


In [13]:
saving_response

{'100': ['\nA:Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. For example, some images may show more trees or less trees due to seasonal changes, weather patterns, or human activity. By comparing the images side by side, one can observe these differences and identify any ephemeral gully appearances that were present at different times but not consistently across all images.',
  'Yes'],
 '1004': ['\nA:Yes, there are ephemeral gully appearances when looking at all of them together. The reason is that each image captures a different moment in time, and these moments can show changes in the landscape over the 10-year period. These changes could be due to factors such as weather patterns, soil erosion, or human activity. By comparing the images side by side, one can observe how the area has evolved over time, reveali

In [14]:
saving_response_reformat = {}
for key, item in saving_response.items():
    saving_response_reformat[key] = item[-1]
saving_response_reformat

{'100': 'Yes',
 '1004': 'Yes',
 '1005': 'Yes',
 '1007': 'Yes',
 '1010': 'Yes',
 '1015': 'Yes',
 '1017': 'Yes',
 '1018': 'Yes',
 '1020': 'Yes',
 '1021': 'Yes',
 '1031': 'Yes',
 '1032': 'Yes',
 '1035': 'Yes',
 '1038': 'Yes',
 '104': 'Yes',
 '1040': 'Yes',
 '1041': 'Yes',
 '1043': 'Yes',
 '1049': 'Yes',
 '105': 'Yes',
 '1050': 'Yes',
 '1051': 'Yes',
 '1053': 'Yes',
 '1062': 'Yes',
 '1064': 'Yes',
 '1065': 'Yes',
 '1067': 'Yes',
 '107': 'Yes',
 '1073': 'Yes',
 '1078': 'Yes',
 '1081': 'Yes',
 '1085': 'Yes',
 '1095': 'Yes',
 '1098': 'Yes',
 '1099': 'Yes',
 '1114': 'Yes',
 '1117': 'Yes',
 '1121': 'Yes',
 '1123': 'Yes',
 '1128': 'Yes',
 '1133': 'Yes',
 '1137': 'Yes',
 '1138': 'Yes',
 '1140': 'Yes',
 '1141': 'Yes',
 '1143': 'Yes',
 '117': 'Yes',
 '118': 'Yes',
 '123': 'Yes',
 '124': 'Yes',
 '125': 'Yes',
 '126': 'Yes',
 '128': 'Yes',
 '129': 'Yes',
 '131': 'Yes',
 '135': 'Yes',
 '248': 'Yes',
 '250': 'Yes',
 '251': 'Yes',
 '253': 'Yes',
 '266': 'Yes',
 '267': 'Yes',
 '272': 'Yes',
 '273': 'Yes'

In [15]:
saving_response_final = copy.deepcopy(saving_response_reformat)

for key, item in saving_response_reformat.items():
    if 'yes' in item.lower():
        saving_response_final[key] = ['4']
    if 'no' in item.lower():
        saving_response_final[key] = ['0']

saving_response_final

{'100': ['4'],
 '1004': ['4'],
 '1005': ['4'],
 '1007': ['4'],
 '1010': ['4'],
 '1015': ['4'],
 '1017': ['4'],
 '1018': ['4'],
 '1020': ['4'],
 '1021': ['4'],
 '1031': ['4'],
 '1032': ['4'],
 '1035': ['4'],
 '1038': ['4'],
 '104': ['4'],
 '1040': ['4'],
 '1041': ['4'],
 '1043': ['4'],
 '1049': ['4'],
 '105': ['4'],
 '1050': ['4'],
 '1051': ['4'],
 '1053': ['4'],
 '1062': ['4'],
 '1064': ['4'],
 '1065': ['4'],
 '1067': ['4'],
 '107': ['4'],
 '1073': ['4'],
 '1078': ['4'],
 '1081': ['4'],
 '1085': ['4'],
 '1095': ['4'],
 '1098': ['4'],
 '1099': ['4'],
 '1114': ['4'],
 '1117': ['4'],
 '1121': ['4'],
 '1123': ['4'],
 '1128': ['4'],
 '1133': ['4'],
 '1137': ['4'],
 '1138': ['4'],
 '1140': ['4'],
 '1141': ['4'],
 '1143': ['4'],
 '117': ['4'],
 '118': ['4'],
 '123': ['4'],
 '124': ['4'],
 '125': ['4'],
 '126': ['4'],
 '128': ['4'],
 '129': ['4'],
 '131': ['4'],
 '135': ['4'],
 '248': ['4'],
 '250': ['4'],
 '251': ['4'],
 '253': ['4'],
 '266': ['4'],
 '267': ['4'],
 '272': ['4'],
 '273': ['4']

In [16]:
# Save the dictionary as a JSON file
filename =  'reason_first_final_results_'+args.modelname+'.json'
with open(os.path.join(args.results_dir, filename), 'w') as json_file:
    json.dump(saving_response_final, json_file, indent=4)

print(f"JSON file {filename} has been created.")

JSON file reason_first_final_results_llava-llama3:8b.json has been created.
